In [1]:
import math
import re
import warnings
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.tools import add_constant
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

from linearmodels import PanelOLS
from linearmodels.panel import compare

import geopandas as gpd
from libpysal.weights import Queen, lag_spatial
from esda.moran import Moran
from spreg import ML_Lag, ML_Error

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
pre_ts_analysis = pd.read_csv('Research_Firearm/pre_ts_analysis_clean.csv')



In [3]:
pre_ts_analysis.columns

Index(['Year', 'Precinct', 'Full Time Positions', 'Budget', 'Borough',
       'MURDER & NON NEGL. MANSLAUGHTER', 'RAPE', 'ROBBERY', 'FELONY ASSAULT',
       'BURGLARY', 'GRAND LARCENY', 'GRAND LARCENY OF MOTOR VEHICLE',
       'TOTAL SEVEN MAJOR FELONY OFFENSES', 'Population_Year'],
      dtype='object')

In [4]:
precinct_counts = pre_ts_analysis['Precinct'].value_counts()
print(precinct_counts.to_string())

Precinct
1      18
68     18
94     18
90     18
88     18
84     18
83     18
81     18
79     18
78     18
77     18
76     18
75     18
73     18
72     18
71     18
70     18
100    18
101    18
102    18
111    18
122    18
120    18
115    18
114    18
113    18
112    18
110    18
103    18
109    18
108    18
107    18
106    18
105    18
104    18
69     18
67     18
5      18
66     18
32     18
30     18
28     18
26     18
25     18
24     18
23     18
20     18
19     18
17     18
13     18
10     18
9      18
7      18
6      18
33     18
34     18
40     18
49     18
63     18
62     18
61     18
60     18
52     18
50     18
48     18
41     18
47     18
46     18
45     18
44     18
43     18
42     18
123    18


In [5]:
pre_ts_analysis

,Year,Precinct,Full Time Positions,Budget,Borough,MURDER & NON NEGL. MANSLAUGHTER,RAPE,ROBBERY,FELONY ASSAULT,BURGLARY,GRAND LARCENY,GRAND LARCENY OF MOTOR VEHICLE,TOTAL SEVEN MAJOR FELONY OFFENSES,Population_Year
0,2006,1,219,11001943,MANHATTAN SOUTH,1,4,119,94,255,1462,78,2013,59431
1,2006,5,240,12354423,MANHATTAN SOUTH,2,5,132,104,152,605,40,1040,53356
2,2006,6,237,10716126,MANHATTAN SOUTH,3,6,214,123,280,1283,70,1979,61259
3,2006,7,174,7786080,MANHATTAN SOUTH,4,7,176,105,127,362,83,864,55703
4,2006,9,234,10190005,MANHATTAN SOUTH,1,14,252,165,297,775,75,1579,76639
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1309,2023,114,252,20752787,QUEENS NORTH,8,40,279,509,224,920,430,2410,210231
1310,2023,115,289,16525730,QUEENS NORTH,4,27,365,481,136,863,337,2213,181382
1311,2023,120,399,29273345,STATEN ISLAND,12,19,163,470,142,414,156,1376,125104
1312,2023,122,249,19873478,STATEN ISLAND,2,8,50,169,100,474,109,912,146208


In [6]:
# Filtering the DataFrame for precincts 6, 76, and 83
filtered_df = pre_ts_analysis[pre_ts_analysis['Precinct'].isin([6, 76, 83])]

# Grouping by 'Precinct' and 'Year' and counting occurrences
precinct_year_counts = filtered_df.groupby(['Precinct', 'Year']).size().reset_index(name='Count')

# Display the DataFrame
print(precinct_year_counts)

    Precinct  Year  Count
0          6  2006      1
1          6  2007      1
2          6  2008      1
3          6  2009      1
4          6  2010      1
5          6  2011      1
6          6  2012      1
7          6  2013      1
8          6  2014      1
9          6  2015      1
10         6  2016      1
11         6  2017      1
12         6  2018      1
13         6  2019      1
14         6  2020      1
15         6  2021      1
16         6  2022      1
17         6  2023      1
18        76  2006      1
19        76  2007      1
20        76  2008      1
21        76  2009      1
22        76  2010      1
23        76  2011      1
24        76  2012      1
25        76  2013      1
26        76  2014      1
27        76  2015      1
28        76  2016      1
29        76  2017      1
30        76  2018      1
31        76  2019      1
32        76  2020      1
33        76  2021      1
34        76  2022      1
35        76  2023      1
36        83  2006      1
37        83

## Define Violent and Crime, also calculate the percentage based crime and budget allocation on precinct level

In [7]:
# Create 'Violent Crime' column
pre_ts_analysis['Violent Crime'] = pre_ts_analysis['FELONY ASSAULT'] + pre_ts_analysis['MURDER & NON NEGL. MANSLAUGHTER'] + pre_ts_analysis['RAPE'] + pre_ts_analysis['ROBBERY']
# Create 'Property Crime' column
pre_ts_analysis['Property Crime'] = pre_ts_analysis['GRAND LARCENY'] + pre_ts_analysis['GRAND LARCENY OF MOTOR VEHICLE'] + pre_ts_analysis['BURGLARY']

In [8]:
# Calculate yearly totals
yearly_totals = pre_ts_analysis.groupby('Year').sum().reset_index()
# List of metrics for which want to calculate yearly totals
metrics = ['Violent Crime', 'Property Crime', 'Full Time Positions']
# Rename columns to *_by_year
yearly_totals = yearly_totals.rename(columns={metric: f"{metric.lower()}_by_year" for metric in metrics})
# Merge yearly totals with merged_data
merged_data = pd.merge(pre_ts_analysis, yearly_totals[['Year'] + [f"{metric.lower()}_by_year" for metric in metrics]], 
                       on='Year', how='left')

# List of metrics for which to calculate yearly totals and per capita rates
metrics = ['Violent Crime', 'Property Crime','Full Time Positions']
# Loop through each metric
for metric in metrics:
    # Compute the per capita rate
    per_capita_col_name = f"{metric}_per_capita"
    merged_data[per_capita_col_name] = merged_data[metric] / merged_data['Population_Year']    
    # Compute the percentage-based rate
    pct_col_name = f"{metric}_pct"
    yearly_col_name = f"{metric.lower()}_by_year"
    merged_data[pct_col_name] = merged_data[metric] / merged_data[yearly_col_name]

In [9]:
pre_ts_analysis = merged_data

In [10]:
# Calculate yearly totals
yearly_totals = pre_ts_analysis.groupby('Year').sum().reset_index()
# List of metrics for which want to calculate yearly totals
metrics = ['MURDER & NON NEGL. MANSLAUGHTER', 'RAPE', 'ROBBERY', 'FELONY ASSAULT', 
    'BURGLARY', 'GRAND LARCENY', 'GRAND LARCENY OF MOTOR VEHICLE', 
    'TOTAL SEVEN MAJOR FELONY OFFENSES', 'Budget','Population_Year']
# Rename columns to *_by_year
yearly_totals = yearly_totals.rename(columns={metric: f"{metric.lower()}_by_year" for metric in metrics})
# Merge yearly totals with merged_data
merged_data = pd.merge(pre_ts_analysis, yearly_totals[['Year'] + [f"{metric.lower()}_by_year" for metric in metrics]], 
                       on='Year', how='left')

# List of metrics for which to calculate yearly totals and per capita rates
metrics = ['MURDER & NON NEGL. MANSLAUGHTER', 'RAPE', 'ROBBERY', 'FELONY ASSAULT', 
    'BURGLARY', 'GRAND LARCENY', 'GRAND LARCENY OF MOTOR VEHICLE', 
    'TOTAL SEVEN MAJOR FELONY OFFENSES', 'Budget','Population_Year']
# Loop through each metric
for metric in metrics:
    # Compute the per capita rate
    per_capita_col_name = f"{metric}_per_capita"
    merged_data[per_capita_col_name] = merged_data[metric] / merged_data['Population_Year']    
    # Compute the percentage-based rate
    pct_col_name = f"{metric}_pct"
    yearly_col_name = f"{metric.lower()}_by_year"
    merged_data[pct_col_name] = merged_data[metric] / merged_data[yearly_col_name]

In [11]:
pre_ts_analysis = merged_data

In [12]:
pre_ts_analysis['Population_Year_pct']

0       0.007676
1       0.006891
2       0.007912
3       0.007194
4       0.009898
          ...   
1309    0.023993
1310    0.020700
1311    0.014278
1312    0.016686
1313    0.011588
Name: Population_Year_pct, Length: 1314, dtype: float64

In [13]:
pre_ts_analysis.groupby('Year')['Population_Year_pct'].sum()

Year
2006    1.0
2007    1.0
2008    1.0
2009    1.0
2010    1.0
2011    1.0
2012    1.0
2013    1.0
2014    1.0
2015    1.0
2016    1.0
2017    1.0
2018    1.0
2019    1.0
2020    1.0
2021    1.0
2022    1.0
2023    1.0
Name: Population_Year_pct, dtype: float64

## Create lag 1 for meature previous one year for crime and budget variable

In [14]:
# Get all columns ending with '_pct' or '_per_capita'
cols_to_lag = [col for col in pre_ts_analysis.columns if col.endswith('_pct') or col.endswith('_per_capita')]

# Loop through the columns and create a lag-1 column for each
for col in cols_to_lag:
    lag_col_name = col + '_lag1'  # Name of the new lag column
    pre_ts_analysis[lag_col_name] = pre_ts_analysis.groupby('Precinct')[col].shift(1)

In [15]:
pre_ts_analysis.columns

Index(['Year', 'Precinct', 'Full Time Positions', 'Budget', 'Borough',
       'MURDER & NON NEGL. MANSLAUGHTER', 'RAPE', 'ROBBERY', 'FELONY ASSAULT',
       'BURGLARY', 'GRAND LARCENY', 'GRAND LARCENY OF MOTOR VEHICLE',
       'TOTAL SEVEN MAJOR FELONY OFFENSES', 'Population_Year', 'Violent Crime',
       'Property Crime', 'violent crime_by_year', 'property crime_by_year',
       'full time positions_by_year', 'Violent Crime_per_capita',
       'Violent Crime_pct', 'Property Crime_per_capita', 'Property Crime_pct',
       'Full Time Positions_per_capita', 'Full Time Positions_pct',
       'murder & non negl. manslaughter_by_year', 'rape_by_year',
       'robbery_by_year', 'felony assault_by_year', 'burglary_by_year',
       'grand larceny_by_year', 'grand larceny of motor vehicle_by_year',
       'total seven major felony offenses_by_year', 'budget_by_year',
       'population_year_by_year', 'MURDER & NON NEGL. MANSLAUGHTER_per_capita',
       'MURDER & NON NEGL. MANSLAUGHTER_pct', 'RA

In [16]:
expanded_df_with_lags = pd.read_csv("expanded_df_with_lags_demo_06_23.csv")


In [17]:
expanded_df_with_lags

,precinct,Year,total_pop,minority_pct,per_capita_income_inflation_adjust,total_pop_lag1,minority_pct_lag1,per_capita_income_lag1
0,1,2006,83293.379121,0.267230,87482.282663,NaN,NaN,NaN
1,5,2006,93739.912088,0.384627,74453.142608,NaN,NaN,NaN
2,6,2006,70587.428571,0.190757,93954.047326,NaN,NaN,NaN
3,7,2006,88347.670330,0.404016,69879.516762,NaN,NaN,NaN
4,9,2006,112501.010989,0.247033,80506.118187,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1363,115,2023,2047.527473,0.513890,34558.424961,10033.153846,0.522893,33555.777314
1364,120,2023,4539.483516,0.077920,32880.827564,13677.076923,0.085590,32549.305142
1365,121,2023,-12066.340659,0.293450,31389.833940,-2129.153846,0.288328,31228.143323
1366,122,2023,-3587.186813,0.187455,42985.665619,6173.923077,0.181993,41685.091045


In [18]:
expanded_df_with_lags.rename(columns={'precinct': 'Precinct'}, inplace=True)


In [19]:
# Step 1: Match dtypes for merge keys
expanded_df_with_lags['Year'] = pre_ts_analysis['Year'].dtype.type(expanded_df_with_lags['Year'])
expanded_df_with_lags['Precinct'] = pre_ts_analysis['Precinct'].dtype.type(expanded_df_with_lags['Precinct'])

# Step 2: Merge (left join to keep all rows from pre_ts_analysis)
pre_ts_analysis = pre_ts_analysis.merge(
    expanded_df_with_lags,
    how='left',
    on=['Year', 'Precinct']
)

print(pre_ts_analysis.head())


   Year  Precinct  Full Time Positions    Budget          Borough  \
0  2006         1                  219  11001943  MANHATTAN SOUTH   
1  2006         5                  240  12354423  MANHATTAN SOUTH   
2  2006         6                  237  10716126  MANHATTAN SOUTH   
3  2006         7                  174   7786080  MANHATTAN SOUTH   
4  2006         9                  234  10190005  MANHATTAN SOUTH   

   MURDER & NON NEGL. MANSLAUGHTER  RAPE  ROBBERY  FELONY ASSAULT  BURGLARY  \
0                                1     4      119              94       255   
1                                2     5      132             104       152   
2                                3     6      214             123       280   
3                                4     7      176             105       127   
4                                1    14      252             165       297   

   ...  Budget_per_capita_lag1  Budget_pct_lag1  \
0  ...                     NaN              NaN   
1  ...  

In [20]:
# Define updated regression formula
regression_formula = (
    "Budget_per_capita ~ Q('Violent Crime_per_capita_lag1') + "
    "Q('Property Crime_per_capita_lag1') + "
    "Budget_per_capita_lag1 + "
    "minority_pct_lag1 + "
    "per_capita_income_lag1 + "  
    "total_pop_lag1"
)

# Run the regression
model = smf.ols(formula=regression_formula, data=pre_ts_analysis).fit()

# Display summary
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     3309.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:21:07   Log-Likelihood:                -5053.5
No. Observations:                1224   AIC:                         1.012e+04
Df Residuals:                    1217   BIC:                         1.016e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                          coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [21]:
cols = [
    'Budget_per_capita',
    'Violent Crime_per_capita_lag1',
    'Property Crime_per_capita_lag1',
    'Budget_per_capita_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]

df = pre_ts_analysis[cols].dropna().copy()

# Standardize predictors and outcome
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=['Budget_per_capita']))
y_scaled = scaler.fit_transform(df[['Budget_per_capita']])

# Convert back to DataFrame with same column names
X_scaled_df = pd.DataFrame(X_scaled, columns=df.columns[1:], index=df.index)
y_scaled_series = pd.Series(y_scaled.flatten(), name='Budget_per_capita', index=df.index)

# Add intercept
X_scaled_df = sm.add_constant(X_scaled_df)

# Fit standardized OLS model
model_std = sm.OLS(y_scaled_series, X_scaled_df).fit()

# Show results
print(model_std.summary())


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     3309.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:21:07   Log-Likelihood:                 8.2928
No. Observations:                1224   AIC:                            -2.586
Df Residuals:                    1217   BIC:                             33.18
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

In [22]:
cols = [
    'Precinct',                
    'Budget_per_capita',
    'Violent Crime_per_capita_lag1',
    'Property Crime_per_capita_lag1',
    'Budget_per_capita_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]

df = pre_ts_analysis[cols].dropna().copy()

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = df.drop(columns=['Precinct', 'Budget_per_capita'])
y = df[['Budget_per_capita']]

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel() 

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=df.index)
y_scaled_series = pd.Series(y_scaled, name='Budget_per_capita', index=df.index)

X_scaled_df = sm.add_constant(X_scaled_df)

model_std = sm.OLS(y_scaled_series, X_scaled_df).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']} 
)

print(model_std.summary())


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     4756.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.23e-90
Time:                        01:21:07   Log-Likelihood:                 8.2928
No. Observations:                1224   AIC:                            -2.586
Df Residuals:                    1217   BIC:                             33.18
Df Model:                           6                                         
Covariance Type:              cluster                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

In [23]:
cols = [
    "Budget_per_capita",
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[cols].dropna().copy()

# Standardize predictors (X) and target (y)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=["Budget_per_capita"]))
y_scaled = scaler.fit_transform(df[["Budget_per_capita"]])  # returns 2D array

# Convert to DataFrame and Series with proper labels
X_scaled_df = pd.DataFrame(X_scaled, columns=cols[1:], index=df.index)
y_scaled_series = pd.Series(y_scaled.flatten(), name="Budget_per_capita", index=df.index)

X_scaled_df = sm.add_constant(X_scaled_df)

# Fit the OLS regression with standardized variables
model_full_crime = sm.OLS(y_scaled_series, X_scaled_df).fit()

print(model_full_crime.summary())


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.943
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     1818.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:21:07   Log-Likelihood:                 15.035
No. Observations:                1224   AIC:                            -6.070
Df Residuals:                    1212   BIC:                             55.25
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [24]:
cols = [
    "Budget_per_capita",
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[["Precinct"] + cols].dropna().copy()

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = df.drop(columns=["Precinct", "Budget_per_capita"])
y = df[["Budget_per_capita"]]

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel()

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=df.index)
y_scaled_ser = pd.Series(y_scaled, name="Budget_per_capita", index=df.index)

# Add intercept
X_scaled_df = sm.add_constant(X_scaled_df)

res = sm.OLS(y_scaled_ser, X_scaled_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["Precinct"]}
)

print(res.summary())



                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.943
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     3727.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.19e-93
Time:                        01:21:08   Log-Likelihood:                 15.035
No. Observations:                1224   AIC:                            -6.070
Df Residuals:                    1212   BIC:                             55.25
Df Model:                          11                                         
Covariance Type:              cluster                                         
                                                      coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [25]:
cols = [
    "Budget_per_capita",
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[["Precinct"] + cols].dropna().copy()

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X = df.drop(columns=["Precinct", "Budget_per_capita"])
y = df[["Budget_per_capita"]]

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y).ravel()

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns, index=df.index)
y_scaled_ser = pd.Series(y_scaled, name="Budget_per_capita", index=df.index)

# Add intercept
X_scaled_df = sm.add_constant(X_scaled_df)

res = sm.OLS(y_scaled_ser, X_scaled_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["Precinct"]}
)

print(res.summary())

                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.943
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     3699.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.88e-92
Time:                        01:21:08   Log-Likelihood:                 12.654
No. Observations:                1224   AIC:                            -3.309
Df Residuals:                    1213   BIC:                             52.90
Df Model:                          10                                         
Covariance Type:              cluster                                         
                                                      coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [26]:
cols = [
    "Budget_per_capita",
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[cols].dropna().copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop(columns=["Budget_per_capita"]))
y_scaled = scaler.fit_transform(df[["Budget_per_capita"]]) 

# Convert to DataFrame and Series with proper labels
X_scaled_df = pd.DataFrame(X_scaled, columns=cols[1:], index=df.index)
y_scaled_series = pd.Series(y_scaled.flatten(), name="Budget_per_capita", index=df.index)

X_scaled_df = sm.add_constant(X_scaled_df)

# Fit the OLS regression with standardized variables
model_full_crime = sm.OLS(y_scaled_series, X_scaled_df).fit()

print(model_full_crime.summary())

                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.943
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     1994.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:21:08   Log-Likelihood:                 12.654
No. Observations:                1224   AIC:                            -3.309
Df Residuals:                    1213   BIC:                             52.90
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                      coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [27]:
X_vars = [
    'Violent Crime_pct_lag1',
    'Property Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_pct'

# Drop missing values
df = pre_ts_analysis[X_vars + [y_var]].dropna().copy()

# Standardize X
scaler = StandardScaler()
X_standardized = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Standardize y
y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = (df[y_var] - y_mean) / y_std
y_standardized.index = df.index  # match index with X

# Add constant (intercept)
X_standardized = sm.add_constant(X_standardized)

# Fit the standardized OLS model
model_standardized = sm.OLS(y_standardized, X_standardized).fit()

# Print summary
print(model_standardized.summary())


                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.909
Method:                 Least Squares   F-statistic:                     2038.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:21:08   Log-Likelihood:                -266.20
No. Observations:                1224   AIC:                             546.4
Df Residuals:                    1217   BIC:                             582.2
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                    1

# Paper Analysis

In [28]:
X_vars = [
    'Violent Crime_pct_lag1',
    'Property Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_pct'

df = pre_ts_analysis[X_vars + [y_var, 'Precinct']].dropna().copy()

# Standardize X
scaler = StandardScaler()
X_standardized = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Standardize y
y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = (df[y_var] - y_mean) / y_std
y_standardized.index = df.index  

# Add constant (intercept)
X_standardized = sm.add_constant(X_standardized)

# Fit the model with clustered standard errors by precinct
model_clustered = sm.OLS(y_standardized, X_standardized).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Print the summary
print(model_clustered.summary())


                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.909
Method:                 Least Squares   F-statistic:                     3348.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           5.54e-85
Time:                        01:21:08   Log-Likelihood:                -266.20
No. Observations:                1224   AIC:                             546.4
Df Residuals:                    1217   BIC:                             582.2
Df Model:                           6                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                    1

In [29]:
cols = [
    "Precinct", 
    "Budget_per_capita",
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[cols].dropna().copy()

# Save precincts for clustering
precinct_ids = df["Precinct"]

# Standardize predictors and response
X_vars = cols[2:] 
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[X_vars])
y_scaled = scaler.fit_transform(df[["Budget_per_capita"]])

X_scaled_df = pd.DataFrame(X_scaled, columns=X_vars, index=df.index)
y_scaled_series = pd.Series(y_scaled.flatten(), name="Budget_per_capita", index=df.index)

X_vif = X_scaled_df.copy()
vif_df = pd.DataFrame()
vif_df["Variable"] = X_vif.columns
vif_df["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print("=== VIF Results ===")
print(vif_df.sort_values("VIF", ascending=False))

# Add intercept
X_scaled_df = sm.add_constant(X_scaled_df)

# Fit model with clustering by precinct
model_clustered = sm.OLS(y_scaled_series, X_scaled_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": precinct_ids}
)

print("\n=== Clustered Standard Errors OLS Summary ===")
print(model_clustered.summary())


=== VIF Results ===
                                           Variable       VIF
4                           ROBBERY_per_capita_lag1  5.188648
2                    FELONY ASSAULT_per_capita_lag1  4.921896
7                            Budget_per_capita_lag1  2.652618
3                          BURGLARY_per_capita_lag1  2.520715
1                              RAPE_per_capita_lag1  2.470722
5                     GRAND LARCENY_per_capita_lag1  2.468585
0   MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1  2.445818
9                            per_capita_income_lag1  2.380523
8                                 minority_pct_lag1  1.886324
6    GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1  1.843906
10                                   total_pop_lag1  1.193997

=== Clustered Standard Errors OLS Summary ===
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.943
Model:                         

In [30]:
cols = [
    "Precinct", 
    "Budget_per_capita",
    'Violent Crime_per_capita_lag1',
    'Property Crime_per_capita_lag1',
    'Budget_per_capita_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]

df = pre_ts_analysis[cols].dropna().copy()

precinct_ids = df["Precinct"]

# Standardize predictors and response
X_vars = cols[2:]  
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[X_vars])
y_scaled = scaler.fit_transform(df[["Budget_per_capita"]])

X_scaled_df = pd.DataFrame(X_scaled, columns=X_vars, index=df.index)
y_scaled_series = pd.Series(y_scaled.flatten(), name="Budget_per_capita", index=df.index)

X_vif = X_scaled_df.copy()
vif_df = pd.DataFrame()
vif_df["Variable"] = X_vif.columns
vif_df["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print("=== VIF Results ===")
print(vif_df.sort_values("VIF", ascending=False))

X_scaled_df = sm.add_constant(X_scaled_df)

model_clustered = sm.OLS(y_scaled_series, X_scaled_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": precinct_ids}
)

print("\n=== Clustered Standard Errors OLS Summary ===")
print(model_clustered.summary())


=== VIF Results ===
                         Variable       VIF
2          Budget_per_capita_lag1  2.027871
4          per_capita_income_lag1  1.833138
3               minority_pct_lag1  1.737612
0   Violent Crime_per_capita_lag1  1.720789
1  Property Crime_per_capita_lag1  1.393977
5                  total_pop_lag1  1.174885

=== Clustered Standard Errors OLS Summary ===
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     4756.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.23e-90
Time:                        01:21:08   Log-Likelihood:                 8.2928
No. Observations:                1224   AIC:                            -2.586
Df Residuals:                    1217   BIC:                            

In [31]:
cols = [
    "Precinct",  # for clustering
    "Budget_per_capita",
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[cols].dropna().copy()

# Cluster IDs
precinct_ids = df["Precinct"]


X_vars = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var = "Budget_per_capita"

# Standardize X
x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(df[X_vars])
X_scaled_df = pd.DataFrame(X_scaled, columns=X_vars, index=df.index)

# Standardize y (mean/std)
y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = ((df[y_var] - y_mean) / y_std).astype(float)
y_standardized.index = df.index


X_for_vif = X_scaled_df.copy()
vif_df = pd.DataFrame({
    "Variable": X_for_vif.columns,
    "VIF": [variance_inflation_factor(X_for_vif.values, i) for i in range(X_for_vif.shape[1])]
}).sort_values("VIF", ascending=False)

print("=== VIF Results (no FE) ===")
print(vif_df)


X_design = sm.add_constant(X_scaled_df).astype(float)

model_clustered_nofe = sm.OLS(y_standardized, X_design).fit(
    cov_type="cluster",
    cov_kwds={"groups": precinct_ids}
)

print("\n=== Clustered SE OLS (No Fixed Effects) ===")
print(model_clustered_nofe.summary())


=== VIF Results (no FE) ===
                         Variable       VIF
2          Budget_per_capita_lag1  2.027871
4          per_capita_income_lag1  1.833138
3               minority_pct_lag1  1.737612
0   Violent Crime_per_capita_lag1  1.720789
1  Property Crime_per_capita_lag1  1.393977
5                  total_pop_lag1  1.174885

=== Clustered SE OLS (No Fixed Effects) ===
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.942
Model:                            OLS   Adj. R-squared:                  0.942
Method:                 Least Squares   F-statistic:                     4756.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.23e-90
Time:                        01:21:09   Log-Likelihood:                 8.7930
No. Observations:                1224   AIC:                            -3.586
Df Residuals:                    1217   BIC:                      

In [32]:
cols = [
    "Precinct",
    "Year",
    "Budget_per_capita",
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]

df = pre_ts_analysis[cols].dropna().copy()

# Save for clustering
precinct_ids = df["Precinct"]

# Year fixed effects (one-way FE)
year_dummies = pd.get_dummies(df["Year"], prefix="Year", drop_first=True, dtype=float)


X_vars = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var = "Budget_per_capita"

x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(df[X_vars])
X_scaled_df = pd.DataFrame(X_scaled, columns=X_vars, index=df.index)

y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = ((df[y_var] - y_mean) / y_std).astype(float)
y_standardized.index = df.index

# Combine standardized X with year FE
X_combined = pd.concat([X_scaled_df, year_dummies], axis=1)


X_for_vif = X_scaled_df.copy()
vif_df = pd.DataFrame({
    "Variable": X_for_vif.columns,
    "VIF": [variance_inflation_factor(X_for_vif.values, i) for i in range(X_for_vif.shape[1])]
}).sort_values("VIF", ascending=False)

print("=== VIF Results (core predictors only; FE excluded) ===")
print(vif_df)

X_combined = sm.add_constant(X_combined).astype(float)

model_clustered_timefe = sm.OLS(y_standardized, X_combined).fit(
    cov_type="cluster",
    cov_kwds={"groups": precinct_ids}
)

print("\n=== Clustered SE OLS with Year FE ===")
print(model_clustered_timefe.summary())


=== VIF Results (core predictors only; FE excluded) ===
                         Variable       VIF
2          Budget_per_capita_lag1  2.027871
4          per_capita_income_lag1  1.833138
3               minority_pct_lag1  1.737612
0   Violent Crime_per_capita_lag1  1.720789
1  Property Crime_per_capita_lag1  1.393977
5                  total_pop_lag1  1.174885

=== Clustered SE OLS with Year FE ===
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.957
Model:                            OLS   Adj. R-squared:                  0.956
Method:                 Least Squares   F-statistic:                     7160.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):          5.78e-110
Time:                        01:21:09   Log-Likelihood:                 191.53
No. Observations:                1224   AIC:                            -337.1
Df Residuals:                    1201   BIC:

In [33]:
X_vars = [
    'Violent Crime_pct_lag1',
    'Property Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_pct'

required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create year dummies and ensure numeric dtype
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
df = pd.concat([df, year_dummies], axis=1)

# Standardize main X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Combine X predictors with year dummies
X_combined = pd.concat([X_base, year_dummies], axis=1)

# Standardize the outcome variable
y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = ((df[y_var] - y_mean) / y_std).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined)

# Force all data to float64
X_combined = X_combined.astype(float)

# Fit OLS with clustered SEs by precinct
model_clustered_timefe = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Display results
print(model_clustered_timefe.summary())






                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.910
Model:                            OLS   Adj. R-squared:                  0.908
Method:                 Least Squares   F-statistic:                     1367.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           1.81e-84
Time:                        01:21:09   Log-Likelihood:                -265.84
No. Observations:                1224   AIC:                             577.7
Df Residuals:                    1201   BIC:                             695.2
Df Model:                          22                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [34]:
X_vars = [
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var = 'Budget_per_capita'

# Load relevant data and drop missing
required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create year dummies and ensure numeric dtype
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
df = pd.concat([df, year_dummies], axis=1)

# Standardize main X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Combine X predictors with year dummies
X_combined = pd.concat([X_base, year_dummies], axis=1)

# Standardize the outcome variable
y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = ((df[y_var] - y_mean) / y_std).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined)

# Force all data to float64
X_combined = X_combined.astype(float)

# Fit OLS with clustered SEs by precinct
model_clustered_timefe = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Display results
print(model_clustered_timefe.summary())



                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.958
Model:                            OLS   Adj. R-squared:                  0.957
Method:                 Least Squares   F-statistic:                     5855.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):          1.66e-108
Time:                        01:21:09   Log-Likelihood:                 197.03
No. Observations:                1224   AIC:                            -338.1
Df Residuals:                    1196   BIC:                            -195.0
Df Model:                          27                                         
Covariance Type:              cluster                                         
                                                      coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [35]:
X_vars = [
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"

]
y_var = 'Budget_per_capita'

# Ensure required columns are present
required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create Year and Precinct dummies
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
precinct_dummies = pd.get_dummies(df['Precinct'], prefix='Precinct', drop_first=True, dtype=float)

# Standardize X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Combine X predictors + year dummies + precinct dummies
X_combined = pd.concat([X_base, year_dummies, precinct_dummies], axis=1)

# Standardize y
y_standardized = ((df[y_var] - df[y_var].mean()) / df[y_var].std()).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined)

# Ensure all are float type
X_combined = X_combined.astype(float)

# Fit model with **clustered SEs by precinct**
model_fe_both = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Print results
print(model_fe_both.summary())

                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.967
Model:                            OLS   Adj. R-squared:                  0.965
Method:                 Least Squares   F-statistic:                     50.81
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.35e-36
Time:                        01:21:10   Log-Likelihood:                 359.35
No. Observations:                1224   AIC:                            -520.7
Df Residuals:                    1125   BIC:                            -14.83
Df Model:                          98                                         
Covariance Type:              cluster                                         
                                                      coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 98, but rank is 27
  warnings.warn('covariance of constraints does not have full '


Start to regornize Vol+ property crime

In [36]:
X_vars = [
    'Violent Crime_per_capita_lag1',
    'Property Crime_per_capita_lag1',
    'Budget_per_capita_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_per_capita'

# Ensure required columns are present
required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create Year dummies only (one-way FE)
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)

# Standardize X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Combine X predictors + year dummies
X_combined = pd.concat([X_base, year_dummies], axis=1)

# Standardize y
y_standardized = ((df[y_var] - df[y_var].mean()) / df[y_var].std()).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined).astype(float)

# Fit model with clustered SEs by precinct
model_fe_year = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Print results
print(model_fe_year.summary())


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.957
Model:                            OLS   Adj. R-squared:                  0.956
Method:                 Least Squares   F-statistic:                     7160.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):          5.78e-110
Time:                        01:21:10   Log-Likelihood:                 191.53
No. Observations:                1224   AIC:                            -337.1
Df Residuals:                    1201   BIC:                            -219.5
Df Model:                          22                                         
Covariance Type:              cluster                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

In [37]:
X_vars = [
    'Violent Crime_per_capita_lag1',
    'Property Crime_per_capita_lag1',
    'Budget_per_capita_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_per_capita'

# Ensure required columns are present
required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create Year and Precinct dummies
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
precinct_dummies = pd.get_dummies(df['Precinct'], prefix='Precinct', drop_first=True, dtype=float)

# Standardize X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

X_combined = pd.concat([X_base, year_dummies, precinct_dummies], axis=1)

# Standardize y
y_standardized = ((df[y_var] - df[y_var].mean()) / df[y_var].std()).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined)

# Ensure all are float type
X_combined = X_combined.astype(float)

model_fe_both = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Print results
print(model_fe_both.summary())

                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.967
Model:                            OLS   Adj. R-squared:                  0.964
Method:                 Least Squares   F-statistic:                     41.60
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           4.74e-32
Time:                        01:21:10   Log-Likelihood:                 354.08
No. Observations:                1224   AIC:                            -520.2
Df Residuals:                    1130   BIC:                            -39.83
Df Model:                          93                                         
Covariance Type:              cluster                                         
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 93, but rank is 22
  warnings.warn('covariance of constraints does not have full '


In [38]:
X_vars = [
    'Violent Crime_pct_lag1',
    'Property Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_pct'

# Ensure required columns are present
required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create Year and Precinct dummies
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
precinct_dummies = pd.get_dummies(df['Precinct'], prefix='Precinct', drop_first=True, dtype=float)

# Standardize X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Combine X predictors + year dummies + precinct dummies
X_combined = pd.concat([X_base, year_dummies, precinct_dummies], axis=1)

# Standardize y
y_standardized = ((df[y_var] - df[y_var].mean()) / df[y_var].std()).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined)

# Ensure all are float type
X_combined = X_combined.astype(float)

model_fe_both = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

# Print results
print(model_fe_both.summary())


                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.928
Model:                            OLS   Adj. R-squared:                  0.922
Method:                 Least Squares   F-statistic:                     20.50
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           1.54e-22
Time:                        01:21:10   Log-Likelihood:                -126.77
No. Observations:                1224   AIC:                             441.5
Df Residuals:                    1130   BIC:                             921.9
Df Model:                          93                                         
Covariance Type:              cluster                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 93, but rank is 22
  warnings.warn('covariance of constraints does not have full '


# Spatial Analysis

In [39]:
# Load GeoJSON
precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson")

# Check CRS
precincts_gdf_s = precincts_gdf_s.to_crs(epsg=4326)

In [40]:
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})


In [41]:
precincts_gdf_s

,FID,shape_area,shape_leng,Precinct,geometry
0,1,4.730176e+07,80586.154615,1,"MULTIPOLYGON (((-74.04388 40.69019, -74.04351 ..."
1,2,1.808880e+07,18676.124259,5,"POLYGON ((-73.98864 40.72293, -73.98869 40.722..."
2,3,2.213193e+07,27182.610113,6,"POLYGON ((-73.99968 40.73855, -73.99684 40.737..."
3,4,1.836402e+07,17301.308682,7,"POLYGON ((-73.97345 40.71896, -73.97351 40.718..."
4,5,2.139423e+07,19773.233410,9,"POLYGON ((-73.97161 40.72672, -73.97163 40.726..."
...,...,...,...,...,...
72,73,1.132939e+08,58272.204348,115,"POLYGON ((-73.85908 40.76252, -73.85943 40.762..."
73,74,2.325353e+08,96171.721461,120,"POLYGON ((-74.05357 40.60370, -74.05407 40.603..."
74,75,4.757161e+08,138115.721207,121,"MULTIPOLYGON (((-74.15946 40.64145, -74.15975 ..."
75,76,4.547993e+08,154881.006310,122,"MULTIPOLYGON (((-74.05051 40.56642, -74.05047 ..."


In [42]:
merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")

# Create spatial weights
from libpysal.weights import Queen, lag_spatial
w = Queen.from_dataframe(merged_sp)
w.transform = "r"



C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\2348201342.py:5: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


In [43]:
precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

# Merge
merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")

# Clean column names for regression formula compatibility
merged_sp.columns = [col.replace(" ", "_") for col in merged_sp.columns]

# Filter out 2006 (first lag year)
merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

# Define variables
X_vars = [
    'Violent_Crime_pct_lag1',
    'Property_Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_pct'

# Create spatial weights
w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

# Create spatial lag of outcome
merged_sp['lag_Budget_pct'] = lag_spatial(w, merged_sp[y_var])

# Fill missing X values (if any) before spatial lagging
for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())
    merged_sp[f'lag_{var}'] = lag_spatial(w, merged_sp[var])

# Drop rows where spatial lags are still NaN (disconnected precincts only)
lag_vars = [f'lag_{var}' for var in X_vars] + ['lag_Budget_pct']
merged_sp_clean = merged_sp.dropna(subset=[y_var] + lag_vars)

# Confirm number of usable rows
print("Original (post-2006):", len(merged_sp))
print("Rows after dropna (usable for regression):", len(merged_sp_clean))

# Build regression formula
formula = f"{y_var} ~ lag_Budget_pct + " + " + ".join(X_vars + [f'lag_{var}' for var in X_vars])

# Fit model
model = smf.ols(formula=formula, data=merged_sp_clean).fit()
print(model.summary())

C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\2642333130.py:25: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Original (post-2006): 1241
Rows after dropna (usable for regression): 1241
                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.917
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     1039.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:21:54   Log-Likelihood:                 6865.2
No. Observations:                1241   AIC:                        -1.370e+04
Df Residuals:                    1227   BIC:                        -1.363e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------

In [44]:
# Load precinct shapefile and merge with panel data
precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})
merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")

# Clean column names
merged_sp.columns = [col.replace(" ", "_") for col in merged_sp.columns]

merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

# Define independent variables (keep all original X_vars)
X_vars = [
    'Violent_Crime_pct_lag1',
    'Property_Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]

# Define spatial lag variables (exclude lag_Budget_pct)
spatial_lags = [
    'lag_Violent_Crime_pct_lag1',
    'lag_Property_Crime_pct_lag1',
    'lag_Budget_pct_lag1',
    'lag_minority_pct_lag1',
    'lag_per_capita_income_lag1',
    'lag_total_pop_lag1'
]

y_var = 'Budget_pct'

# Create spatial weights matrix
w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

# Fill missing values in predictors (optional, safe for spatial lagging)
for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())

# Compute spatial lags (except lag_Budget_pct)
for var in X_vars:
    merged_sp[f'lag_{var}'] = lag_spatial(w, merged_sp[var])

# Drop rows with any missing value in outcome or predictors
merged_sp_clean = merged_sp.dropna(subset=[y_var] + X_vars + spatial_lags)

# Build regression formula (excluding lag_Budget_pct)
formula = f"{y_var} ~ " + " + ".join(X_vars + spatial_lags)

# Fit model
model = smf.ols(formula=formula, data=merged_sp_clean).fit()
print(model.summary())

C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\1746595464.py:34: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.913
Model:                            OLS   Adj. R-squared:                  0.912
Method:                 Least Squares   F-statistic:                     1074.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:22:15   Log-Likelihood:                 6838.3
No. Observations:                1241   AIC:                        -1.365e+04
Df Residuals:                    1228   BIC:                        -1.358e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [45]:
for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())

In [46]:
# Define predictors
X_vars = [
    'Violent Crime_pct_lag1',
    'Property Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]



In [47]:
# Clean column names: replace spaces with underscores
merged_sp.columns = [col.replace(" ", "_") for col in merged_sp.columns]

# Define cleaned predictors and dependent variable
X_vars = [
    'Violent_Crime_pct_lag1',
    'Property_Crime_pct_lag1',
    'Budget_pct_lag1',
    'minority_pct_lag1',
    'per_capita_income_lag1',
    'total_pop_lag1'
]
y_var = 'Budget_pct'

# Create spatial weights matrix
w = Queen.from_dataframe(merged_sp)
w.transform = 'r'  # Row-standardized

# Add spatial lag of the dependent variable
merged_sp['lag_Budget_pct'] = lag_spatial(w, merged_sp[y_var])

# Add spatial lags of all predictor variables
for var in X_vars:
    merged_sp[f'lag_{var}'] = lag_spatial(w, merged_sp[var])

# Drop rows with missing values in predictors or lags (e.g., disconnected precincts)
all_predictors = ['lag_Budget_pct'] + X_vars + [f'lag_{var}' for var in X_vars]
merged_sp_clean = merged_sp.dropna(subset=[y_var] + all_predictors)

# Build the formula for regression
formula = f"{y_var} ~ lag_Budget_pct + " + " + ".join(X_vars + [f'lag_{var}' for var in X_vars])

# Fit the OLS model
model = smf.ols(formula=formula, data=merged_sp_clean).fit()

print(model.summary())


C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\563835111.py:16: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


                            OLS Regression Results                            
Dep. Variable:             Budget_pct   R-squared:                       0.917
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     1039.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):               0.00
Time:                        01:22:34   Log-Likelihood:                 6865.2
No. Observations:                1241   AIC:                        -1.370e+04
Df Residuals:                    1227   BIC:                        -1.363e+04
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
Intercept         

In [48]:
X_vars = [
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var = 'Budget_per_capita'

required_columns = X_vars + [y_var, 'Precinct', 'Year']
df = pre_ts_analysis[required_columns].dropna().copy()

# Create year dummies and ensure numeric dtype
year_dummies = pd.get_dummies(df['Year'], prefix='Year', drop_first=True, dtype=float)
df = pd.concat([df, year_dummies], axis=1)

# Standardize main X predictors
scaler = StandardScaler()
X_base = pd.DataFrame(
    scaler.fit_transform(df[X_vars]),
    columns=X_vars,
    index=df.index
)

# Combine X predictors with year dummies
X_combined = pd.concat([X_base, year_dummies], axis=1)

# Standardize the outcome variable
y_mean = df[y_var].mean()
y_std = df[y_var].std()
y_standardized = ((df[y_var] - y_mean) / y_std).astype(float)
y_standardized.index = df.index

# Add intercept
X_combined = sm.add_constant(X_combined)

# Force all data to float64
X_combined = X_combined.astype(float)

# Fit OLS with clustered SEs by precinct
model_clustered_timefe = sm.OLS(y_standardized, X_combined).fit(
    cov_type='cluster',
    cov_kwds={'groups': df['Precinct']}
)

print(model_clustered_timefe.summary())


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.958
Model:                            OLS   Adj. R-squared:                  0.957
Method:                 Least Squares   F-statistic:                     5855.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):          1.66e-108
Time:                        01:22:34   Log-Likelihood:                 197.03
No. Observations:                1224   AIC:                            -338.1
Df Residuals:                    1196   BIC:                            -195.0
Df Model:                          27                                         
Covariance Type:              cluster                                         
                                                      coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------

In [49]:
def clean_name(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")

# clean all column names for formula safety
merged_sp = clean_cols(merged_sp)

# filter out first lag year
merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

X_vars_raw = [
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1",
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var  = clean_name(y_var_raw)

# -------- spatial weights (Queen contiguity, row-standardized) --------
w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

# spatial lag of outcome
lag_y = f"lag_{y_var}"
merged_sp[lag_y] = lag_spatial(w, merged_sp[y_var])

for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())
    merged_sp[f"lag_{var}"] = lag_spatial(w, merged_sp[var])

lag_vars = [f"lag_{v}" for v in X_vars] + [lag_y]
merged_sp_clean = merged_sp.dropna(subset=[y_var] + lag_vars).copy()

print("Original (post-2006):", len(merged_sp))
print("Rows after dropna (usable for regression):", len(merged_sp_clean))

# -------- formula: SDM (Wy + X + WX) --------
rhs = [lag_y] + X_vars + [f"lag_{v}" for v in X_vars]
formula = f"{y_var} ~ " + " + ".join(rhs)
print("Formula:", formula)


model_cluster = smf.ols(formula=formula, data=merged_sp_clean).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_sp_clean["Precinct"]}
)
print(model_cluster.summary())



C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\1994377868.py:39: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Original (post-2006): 1241
Rows after dropna (usable for regression): 1241
Formula: Budget_per_capita ~ lag_Budget_per_capita + MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1 + RAPE_per_capita_lag1 + FELONY_ASSAULT_per_capita_lag1 + BURGLARY_per_capita_lag1 + GRAND_LARCENY_per_capita_lag1 + GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1 + ROBBERY_per_capita_lag1 + Budget_per_capita_lag1 + minority_pct_lag1 + per_capita_income_lag1 + total_pop_lag1 + lag_MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1 + lag_RAPE_per_capita_lag1 + lag_FELONY_ASSAULT_per_capita_lag1 + lag_BURGLARY_per_capita_lag1 + lag_GRAND_LARCENY_per_capita_lag1 + lag_GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1 + lag_ROBBERY_per_capita_lag1 + lag_Budget_per_capita_lag1 + lag_minority_pct_lag1 + lag_per_capita_income_lag1 + lag_total_pop_lag1
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.962
Model:                  

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 23, but rank is 19
  warnings.warn('covariance of constraints does not have full '


In [50]:
def clean_name(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")
merged_sp = clean_cols(merged_sp)

merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

X_vars_raw = [
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var = clean_name(y_var_raw)

w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

# Lag of outcome
lag_y = f"lag_{y_var}"
merged_sp[lag_y] = lag_spatial(w, merged_sp[y_var])

# Lags of predictors
for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())
    merged_sp[f"lag_{var}"] = lag_spatial(w, merged_sp[var])

# Drop rows with missing spatial lags
lag_vars = [f"lag_{v}" for v in X_vars] + [lag_y]
merged_sp_clean = merged_sp.dropna(subset=[y_var] + lag_vars).copy()

year_dummies = pd.get_dummies(merged_sp_clean['Year'], prefix='Year', drop_first=True, dtype=float)
merged_sp_clean = pd.concat([merged_sp_clean, year_dummies], axis=1)

rhs = [lag_y] + X_vars + [f"lag_{v}" for v in X_vars] + list(year_dummies.columns)
formula = f"{y_var} ~ " + " + ".join(rhs)
print("Formula:", formula)

model_timefe_cluster = smf.ols(formula=formula, data=merged_sp_clean).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_sp_clean["Precinct"]}
)

print(model_timefe_cluster.summary())


C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\2406526920.py:35: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Formula: Budget_per_capita ~ lag_Budget_per_capita + MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1 + RAPE_per_capita_lag1 + FELONY_ASSAULT_per_capita_lag1 + BURGLARY_per_capita_lag1 + GRAND_LARCENY_per_capita_lag1 + GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1 + ROBBERY_per_capita_lag1 + Budget_per_capita_lag1 + minority_pct_lag1 + per_capita_income_lag1 + total_pop_lag1 + lag_MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1 + lag_RAPE_per_capita_lag1 + lag_FELONY_ASSAULT_per_capita_lag1 + lag_BURGLARY_per_capita_lag1 + lag_GRAND_LARCENY_per_capita_lag1 + lag_GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1 + lag_ROBBERY_per_capita_lag1 + lag_Budget_per_capita_lag1 + lag_minority_pct_lag1 + lag_per_capita_income_lag1 + lag_total_pop_lag1 + Year_2008 + Year_2009 + Year_2010 + Year_2011 + Year_2012 + Year_2013 + Year_2014 + Year_2015 + Year_2016 + Year_2017 + Year_2018 + Year_2019 + Year_2020 + Year_2021 + Year_2022 + Year_2023
                            OLS Regression Results               

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 39, but rank is 34
  warnings.warn('covariance of constraints does not have full '


In [51]:
def clean_name(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")
merged = clean_cols(merged)              # clean all column names
merged = merged[merged["Year"] > 2006].copy()

X_vars_raw = [
    "MURDER & NON NEGL. MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND LARCENY_per_capita_lag1",
    "GRAND LARCENY OF MOTOR VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var  = clean_name(y_var_raw)

keep_cols = ["Precinct", "Year", y_var] + X_vars
df = merged[keep_cols].dropna().copy()

w = Queen.from_dataframe(merged.loc[df.index, ["geometry"]].join(df[["Precinct","Year"]]))
w.transform = 'r'

lag_y = f"lag_{y_var}"
df[lag_y] = lag_spatial(w, df[y_var])

for var in X_vars:
    df[var] = df[var].fillna(df[var].mean())
    df[f"lag_{var}"] = lag_spatial(w, df[var])

lag_cols = [lag_y] + [f"lag_{v}" for v in X_vars]
df = df.dropna(subset=[y_var] + lag_cols).copy()

year_dummies = pd.get_dummies(df["Year"], prefix="Year", drop_first=True, dtype=float)
prec_dummies = pd.get_dummies(df["Precinct"], prefix="Precinct", drop_first=True, dtype=float)

scaler_X = StandardScaler()
scaler_y = StandardScaler()

cont_X_cols = X_vars + lag_cols  
X_cont_scaled = pd.DataFrame(
    scaler_X.fit_transform(df[cont_X_cols]),
    columns=cont_X_cols,
    index=df.index
)

y_scaled = scaler_y.fit_transform(df[[y_var]]).ravel()
y_scaled = pd.Series(y_scaled, name=y_var, index=df.index)

# ---------------- Design matrix ----------------
X = pd.concat([X_cont_scaled, year_dummies, prec_dummies], axis=1)
X = sm.add_constant(X).astype(float)

# ---------------- Fit OLS with clustered SEs by Precinct ----------------
res = sm.OLS(y_scaled, X).fit(
    cov_type="cluster",
    cov_kwds={"groups": df["Precinct"]}  # one-way clustering by unit
)

print(res.summary())

coef_table = (
    pd.DataFrame({
        "beta_std": res.params,
        "se_cluster": res.bse,
        "z": res.tvalues,
        "pval": res.pvalues
    })
    .assign(
        is_dummy=lambda s: s.index.str.startswith(("Year_", "Precinct_")) | (s.index == "const")
    )
)
print("\nStandardized coefficients (continuous vars only), sorted by |beta|:\n")
print(coef_table.loc[~coef_table["is_dummy"]]
      .drop(columns="is_dummy")
      .sort_values("beta_std", key=lambda s: s.abs(), ascending=False)
      .round(4))


C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\1555611789.py:37: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged.loc[df.index, ["geometry"]].join(df[["Precinct","Year"]]))
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 5 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.993
Model:                            OLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                     166.9
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           1.67e-56
Time:                        01:23:38   Log-Likelihood:                 1320.4
No. Observations:                1224   AIC:                            -2419.
Df Residuals:                    1113   BIC:                            -1852.
Df Model:                         110                                         
Covariance Type:              cluster                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 110, but rank is 39
  warnings.warn('covariance of constraints does not have full '


In [52]:
def clean_name(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")
merged_sp = clean_cols(merged_sp)

merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

X_vars_raw = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var = clean_name(y_var_raw)

w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

# Lag of outcome
lag_y = f"lag_{y_var}"
merged_sp[lag_y] = lag_spatial(w, merged_sp[y_var])

# Lags of predictors
for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())
    merged_sp[f"lag_{var}"] = lag_spatial(w, merged_sp[var])

lag_vars = [f"lag_{v}" for v in X_vars] + [lag_y]
merged_sp_clean = merged_sp.dropna(subset=[y_var] + lag_vars).copy()

year_dummies = pd.get_dummies(
    merged_sp_clean['Year'], 
    prefix='Year', 
    drop_first=True, 
    dtype=float
)
merged_sp_clean = pd.concat([merged_sp_clean, year_dummies], axis=1)

rhs = [lag_y] + X_vars + [f"lag_{v}" for v in X_vars] + list(year_dummies.columns)
formula = f"{y_var} ~ " + " + ".join(rhs)
print("Formula:", formula)

model_timefe_cluster = smf.ols(formula=formula, data=merged_sp_clean).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_sp_clean["Precinct"]}
)

print(model_timefe_cluster.summary())


C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\1872542977.py:30: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Formula: Budget_per_capita ~ lag_Budget_per_capita + Violent_Crime_per_capita_lag1 + Property_Crime_per_capita_lag1 + Budget_per_capita_lag1 + minority_pct_lag1 + per_capita_income_lag1 + total_pop_lag1 + lag_Violent_Crime_per_capita_lag1 + lag_Property_Crime_per_capita_lag1 + lag_Budget_per_capita_lag1 + lag_minority_pct_lag1 + lag_per_capita_income_lag1 + lag_total_pop_lag1 + Year_2008 + Year_2009 + Year_2010 + Year_2011 + Year_2012 + Year_2013 + Year_2014 + Year_2015 + Year_2016 + Year_2017 + Year_2018 + Year_2019 + Year_2020 + Year_2021 + Year_2022 + Year_2023
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.964
Model:                            OLS   Adj. R-squared:                  0.963
Method:                 Least Squares   F-statistic:                     896.4
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           5.34e-80
Time:                        01:24

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 29, but rank is 25
  warnings.warn('covariance of constraints does not have full '


In [53]:
def clean_name(s: str) -> str:
    """Clean column names for formula safety"""
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")
merged_sp = clean_cols(merged_sp)

merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

X_vars_raw = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var = clean_name(y_var_raw)

w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

# Lag of outcome
lag_y = f"lag_{y_var}"
merged_sp[lag_y] = lag_spatial(w, merged_sp[y_var])

# Lags of predictors
for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())
    merged_sp[f"lag_{var}"] = lag_spatial(w, merged_sp[var])

lag_vars = [f"lag_{v}" for v in X_vars] + [lag_y]
merged_sp_clean = merged_sp.dropna(subset=[y_var] + lag_vars + ['Year','Precinct']).copy()

year_dummies = pd.get_dummies(
    merged_sp_clean['Year'],
    prefix='Year',
    drop_first=True,
    dtype=float
)
merged_sp_clean = pd.concat([merged_sp_clean, year_dummies], axis=1)

cont_cols = [y_var] + X_vars + lag_vars
scaler = StandardScaler()
merged_sp_clean[[c + "_z" for c in cont_cols]] = scaler.fit_transform(merged_sp_clean[cont_cols])

y_var_z   = y_var + "_z"
lag_y_z   = lag_y + "_z"
X_vars_z  = [v + "_z" for v in X_vars]
lag_vars_z= [lv + "_z" for lv in lag_vars]

rhs = [lag_y_z] + X_vars_z + lag_vars_z + list(year_dummies.columns)
formula = f"{y_var_z} ~ " + " + ".join(rhs)
print("Formula:", formula)

model_timefe_cluster = smf.ols(formula=formula, data=merged_sp_clean).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_sp_clean["Precinct"]}
)

print(model_timefe_cluster.summary())

coef_series = model_timefe_cluster.params
se_series   = model_timefe_cluster.bse
t_series    = model_timefe_cluster.tvalues
p_series    = model_timefe_cluster.pvalues

coef_table = (
    pd.DataFrame({
        "beta_std": coef_series,
        "se_cluster": se_series,
        "z": t_series,
        "pval": p_series
    })
    .assign(is_dummy=lambda s: s.index.str.startswith("Year_") | (s.index=="Intercept"))
)

print("\nStandardized coefficients (continuous vars only), sorted by |beta|:\n")
print(
    coef_table.loc[~coef_table["is_dummy"]]
              .drop(columns="is_dummy")
              .sort_values("beta_std", key=lambda s: s.abs(), ascending=False)
              .round(4)
)


def back_transform_beta(beta_std: float, x_name: str) -> float:
    """
    Convert standardized beta back to original units for predictor x_name.
    beta_unstd = beta_std * (sd_Y / sd_X)
    """
    # map names to scaler columns
    col_map = {c+"_z": i for i, c in enumerate(cont_cols)}
    sd_Y = scaler.scale_[cont_cols.index(y_var)]
    sd_X = scaler.scale_[cont_cols.index(x_name)]
    return float(beta_std * (sd_Y / sd_X))




C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\4189272711.py:31: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Formula: Budget_per_capita_z ~ lag_Budget_per_capita_z + Violent_Crime_per_capita_lag1_z + Property_Crime_per_capita_lag1_z + Budget_per_capita_lag1_z + minority_pct_lag1_z + per_capita_income_lag1_z + total_pop_lag1_z + lag_Violent_Crime_per_capita_lag1_z + lag_Property_Crime_per_capita_lag1_z + lag_Budget_per_capita_lag1_z + lag_minority_pct_lag1_z + lag_per_capita_income_lag1_z + lag_total_pop_lag1_z + lag_Budget_per_capita_z + Year_2008 + Year_2009 + Year_2010 + Year_2011 + Year_2012 + Year_2013 + Year_2014 + Year_2015 + Year_2016 + Year_2017 + Year_2018 + Year_2019 + Year_2020 + Year_2021 + Year_2022 + Year_2023
                             OLS Regression Results                            
Dep. Variable:     Budget_per_capita_z   R-squared:                       0.964
Model:                             OLS   Adj. R-squared:                  0.963
Method:                  Least Squares   F-statistic:                     1094.
Date:                 Mon, 13 Oct 2025   Prob (F-statis

In [54]:
def clean_name(s: str) -> str:
    """Clean column names for formula safety"""
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")
merged_sp = clean_cols(merged_sp)

merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

X_vars_raw = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var = clean_name(y_var_raw)

w = Queen.from_dataframe(merged_sp)
w.transform = 'r'

lag_y = f"lag_{y_var}"
merged_sp[lag_y] = lag_spatial(w, merged_sp[y_var])

for var in X_vars:
    merged_sp[var] = merged_sp[var].fillna(merged_sp[var].mean())
    merged_sp[f"lag_{var}"] = lag_spatial(w, merged_sp[var])

lag_vars = [f"lag_{v}" for v in X_vars] + [lag_y]
merged_sp_clean = merged_sp.dropna(subset=[y_var] + lag_vars + ['Year','Precinct']).copy()

year_dummies = pd.get_dummies(
    merged_sp_clean['Year'],
    prefix='Year',
    drop_first=True,
    dtype=float
)
merged_sp_clean = pd.concat([merged_sp_clean, year_dummies], axis=1)

cont_cols = [y_var] + X_vars + lag_vars
scaler = StandardScaler()
merged_sp_clean[cont_cols] = scaler.fit_transform(merged_sp_clean[cont_cols])

rhs = [lag_y] + X_vars + lag_vars + list(year_dummies.columns)
formula = f"{y_var} ~ " + " + ".join(rhs)
print("Formula:", formula)

model_timefe_cluster = smf.ols(formula=formula, data=merged_sp_clean).fit(
    cov_type="cluster",
    cov_kwds={"groups": merged_sp_clean["Precinct"]}
)

print(model_timefe_cluster.summary())

coef_table = (
    pd.DataFrame({
        "beta_std": model_timefe_cluster.params,
        "se_cluster": model_timefe_cluster.bse,
        "z": model_timefe_cluster.tvalues,
        "pval": model_timefe_cluster.pvalues
    })
    .assign(is_dummy=lambda s: s.index.str.startswith("Year_") | (s.index=="Intercept"))
)

print("\nStandardized coefficients (continuous vars only), sorted by |beta|:\n")
print(
    coef_table.loc[~coef_table["is_dummy"]]
              .drop(columns="is_dummy")
              .sort_values("beta_std", key=lambda s: s.abs(), ascending=False)
              .round(4)
)


C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\1581840394.py:31: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(merged_sp)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Formula: Budget_per_capita ~ lag_Budget_per_capita + Violent_Crime_per_capita_lag1 + Property_Crime_per_capita_lag1 + Budget_per_capita_lag1 + minority_pct_lag1 + per_capita_income_lag1 + total_pop_lag1 + lag_Violent_Crime_per_capita_lag1 + lag_Property_Crime_per_capita_lag1 + lag_Budget_per_capita_lag1 + lag_minority_pct_lag1 + lag_per_capita_income_lag1 + lag_total_pop_lag1 + lag_Budget_per_capita + Year_2008 + Year_2009 + Year_2010 + Year_2011 + Year_2012 + Year_2013 + Year_2014 + Year_2015 + Year_2016 + Year_2017 + Year_2018 + Year_2019 + Year_2020 + Year_2021 + Year_2022 + Year_2023
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.964
Model:                            OLS   Adj. R-squared:                  0.963
Method:                 Least Squares   F-statistic:                     1094.
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           2.86e-84
Time:     

In [55]:
def clean_name(s: str) -> str:
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf = precincts_gdf.rename(columns={"precinct": "Precinct"})

merged = precincts_gdf.merge(pre_ts_analysis, on="Precinct")
merged = clean_cols(merged)
merged = merged[merged["Year"] > 2006].copy()

X_vars_raw = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1",
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var  = clean_name(y_var_raw)

keep_cols = ["Precinct", "Year", "geometry", y_var] + X_vars
df = merged[keep_cols].dropna().copy()


w = Queen.from_dataframe(df)
w.transform = 'r'

lag_y = f"lag_{y_var}"
df[lag_y] = lag_spatial(w, df[y_var])

for v in X_vars:
    if df[v].isna().any():
        df[v] = df[v].fillna(df[v].mean())
    df[f"lag_{v}"] = lag_spatial(w, df[v])

lag_cols = [lag_y] + [f"lag_{v}" for v in X_vars]
df = df.dropna(subset=[y_var] + lag_cols).copy()

year_dummies = pd.get_dummies(df["Year"], prefix="Year", drop_first=True, dtype=float)
prec_dummies = pd.get_dummies(df["Precinct"], prefix="Precinct", drop_first=True, dtype=float)


scaler_X = StandardScaler()
scaler_y = StandardScaler()

cont_X_cols = X_vars + lag_cols  

X_cont_scaled = pd.DataFrame(
    scaler_X.fit_transform(df[cont_X_cols]),
    columns=cont_X_cols,
    index=df.index
)

y_scaled = scaler_y.fit_transform(df[[y_var]]).ravel()
y_scaled = pd.Series(y_scaled, name=y_var, index=df.index)

X = pd.concat([X_cont_scaled, year_dummies, prec_dummies], axis=1)
X = sm.add_constant(X).astype(float)


print(res.summary())

coef_table = (
    pd.DataFrame({
        "beta_std": res.params,
        "se_cluster": res.bse,
        "z": res.tvalues,
        "pval": res.pvalues
    })
    .assign(
        is_dummy=lambda s: s.index.str.startswith(("Year_", "Precinct_")) | (s.index == "const")
    )
)

print("\nStandardized coefficients (continuous vars only), sorted by |beta|:\n")
print(
    coef_table.loc[~coef_table["is_dummy"]]
              .drop(columns="is_dummy")
              .sort_values("beta_std", key=lambda s: s.abs(), ascending=False)
              .round(4)
)



C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\194658299.py:33: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(df)


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.993
Model:                            OLS   Adj. R-squared:                  0.993
Method:                 Least Squares   F-statistic:                     166.9
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           1.67e-56
Time:                        01:25:02   Log-Likelihood:                 1320.4
No. Observations:                1224   AIC:                            -2419.
Df Residuals:                    1113   BIC:                            -1852.
Df Model:                         110                                         
Covariance Type:              cluster                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 5 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


In [56]:
def clean_name(s: str) -> str:
    """Clean column names for formula safety"""
    return re.sub(r'[^0-9A-Za-z_]+', '_', s)

def clean_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [clean_name(c) for c in df.columns]
    return df

precincts_gdf_s = gpd.read_file("Geo/NYPD_Precincts.geojson").to_crs(epsg=4326)
precincts_gdf_s = precincts_gdf_s.rename(columns={"precinct": "Precinct"})

merged_sp = precincts_gdf_s.merge(pre_ts_analysis, on="Precinct")
merged_sp = clean_cols(merged_sp)

merged_sp = merged_sp[merged_sp["Year"] > 2006].copy()

X_vars_raw = [
    "Violent Crime_per_capita_lag1",
    "Property Crime_per_capita_lag1",
    "Budget_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1"
]
y_var_raw = "Budget_per_capita"

X_vars = [clean_name(v) for v in X_vars_raw]
y_var  = clean_name(y_var_raw)

# Ensure required cols present
keep_cols = ["Precinct", "Year", "geometry", y_var] + X_vars
df = merged_sp[keep_cols].copy()

w = Queen.from_dataframe(df)
w.transform = 'r'

lag_y = f"lag_{y_var}"
df[lag_y] = lag_spatial(w, df[y_var])

for var in X_vars:
    if df[var].isna().any():
        df[var] = df[var].fillna(df[var].mean())
    df[f"lag_{var}"] = lag_spatial(w, df[var])

lag_vars = [f"lag_{v}" for v in X_vars] + [lag_y]
df_clean = df.dropna(subset=[y_var] + lag_vars + ['Year','Precinct']).copy()

cont_cols = [y_var] + X_vars + lag_vars
scaler = StandardScaler()
df_clean[cont_cols] = scaler.fit_transform(df_clean[cont_cols])

rhs_terms = [lag_y] + X_vars + lag_vars + ["C(Year)", "C(Precinct)"]
formula = f"{y_var} ~ " + " + ".join(rhs_terms)
print("Two-way FE + spillovers formula:\n", formula)

model_twfe_cluster = smf.ols(formula=formula, data=df_clean).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_clean["Precinct"]}
)
print(model_twfe_cluster.summary())

coef_table = (
    pd.DataFrame({
        "beta_std": model_twfe_cluster.params,
        "se_cluster": model_twfe_cluster.bse,
        "z": model_twfe_cluster.tvalues,
        "pval": model_twfe_cluster.pvalues
    })
    .assign(is_fe_or_const=lambda s:
            s.index.str.startswith(("C(Year)[T.", "C(Precinct)[T.")) |
            (s.index == "Intercept"))
)

print("\nStandardized coefficients (continuous vars only), sorted by |beta|:\n")
print(
    coef_table.loc[~coef_table["is_fe_or_const"]]
              .drop(columns="is_fe_or_const")
              .sort_values("beta_std", key=lambda s: s.abs(), ascending=False)
              .round(4)
)


C:\Users\yinwe\AppData\Local\Temp\ipykernel_22260\3129583446.py:35: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = Queen.from_dataframe(df)
C:\Users\yinwe\Work_software\Lib\site-packages\libpysal\weights\contiguity.py:347: UserWarning: The weights matrix is not fully connected: 
 There are 6 disconnected components.
  W.__init__(self, neighbors, ids=ids, **kw)


Two-way FE + spillovers formula:
 Budget_per_capita ~ lag_Budget_per_capita + Violent_Crime_per_capita_lag1 + Property_Crime_per_capita_lag1 + Budget_per_capita_lag1 + minority_pct_lag1 + per_capita_income_lag1 + total_pop_lag1 + lag_Violent_Crime_per_capita_lag1 + lag_Property_Crime_per_capita_lag1 + lag_Budget_per_capita_lag1 + lag_minority_pct_lag1 + lag_per_capita_income_lag1 + lag_total_pop_lag1 + lag_Budget_per_capita + C(Year) + C(Precinct)
                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                     126.8
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           6.13e-51
Time:                        01:25:23   Log-Likelihood:                 1191.5
No. Observations:                1241   AIC:                            -2

C:\Users\yinwe\Work_software\Lib\site-packages\statsmodels\base\model.py:1896: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 101, but rank is 29
  warnings.warn('covariance of constraints does not have full '


In [57]:
res = model_twfe_cluster

def _rename_term(name: str) -> str:
    name = re.sub(r'^C\(Year\)\[T\.(\d+)\]$', r'Year_\1', name)
    name = re.sub(r'^C\(Precinct\)\[T\.(.+)\]$', r'Precinct_\1', name)
    if name == "Intercept":
        name = "const"
    return name

renamed_idx = [ _rename_term(s) for s in res.params.index ]
params_ren = pd.Series(res.params.values, index=renamed_idx, name="coef")
se_ren     = pd.Series(res.bse.values,     index=renamed_idx, name="se")
t_ren      = pd.Series(res.tvalues.values, index=renamed_idx, name="z")
p_ren      = pd.Series(res.pvalues.values, index=renamed_idx, name="pval")

table = pd.concat([params_ren, se_ren, t_ren, p_ren], axis=1)
table["is_FE"] = table.index.str.startswith(("Year_", "Precinct_"))

preferred_order = [
    "const",
    # Core outcome spillover + own lag(s)
    "lag_Budget_per_capita",
    "Budget_per_capita_lag1",
    "lag_Budget_per_capita_lag1",
    # Disaggregated index crimes 
    "MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY_ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND_LARCENY_per_capita_lag1",
    "GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    # Aggregated crimes 
    "Violent_Crime_per_capita_lag1",
    "Property_Crime_per_capita_lag1",
    # Socioeconomics
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1",
    # Spatial lags of X
    "lag_MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1",
    "lag_RAPE_per_capita_lag1",
    "lag_FELONY_ASSAULT_per_capita_lag1",
    "lag_BURGLARY_per_capita_lag1",
    "lag_GRAND_LARCENY_per_capita_lag1",
    "lag_GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1",
    "lag_ROBBERY_per_capita_lag1",
    "lag_Violent_Crime_per_capita_lag1",
    "lag_Property_Crime_per_capita_lag1",
    "lag_minority_pct_lag1",
    "lag_per_capita_income_lag1",
    "lag_total_pop_lag1",
]

substantive = table.loc[~table["is_FE"]].drop(columns="is_FE")

present_pref = [v for v in preferred_order if v in substantive.index]
remaining    = [v for v in substantive.index if v not in present_pref]
if remaining:
    remaining_sorted = substantive.loc[remaining].assign(_abs=lambda s: s["coef"].abs()) \
                                             .sort_values("_abs", ascending=False) \
                                             .drop(columns="_abs").index.tolist()
else:
    remaining_sorted = []

ordered_index = present_pref + remaining_sorted
clean_table = substantive.loc[ordered_index].round(4)

print("\n=== TWFE with Spillovers — Substantive Coefficients (FE hidden) ===")
print(clean_table.to_string())

n_year_fe = (table.index.str.startswith("Year_")).sum()
n_prec_fe = (table.index.str.startswith("Precinct_")).sum()
print(f"\n[Note] Hidden fixed effects: {n_year_fe} Year FEs, {n_prec_fe} Precinct FEs.")





=== TWFE with Spillovers — Substantive Coefficients (FE hidden) ===
                                       coef      se        z    pval
const                               29.3907  8.6410   3.4013  0.0007
lag_Budget_per_capita              -35.8373  5.3245  -6.7307  0.0000
Budget_per_capita_lag1               0.5428  0.0497  10.9237  0.0000
lag_Budget_per_capita_lag1          19.7627  4.0296   4.9044  0.0000
Violent_Crime_per_capita_lag1        0.0046  0.0436   0.1062  0.9155
Property_Crime_per_capita_lag1      -0.0500  0.0364  -1.3711  0.1703
minority_pct_lag1                   -0.0388  0.0208  -1.8675  0.0618
per_capita_income_lag1              -0.0190  0.0239  -0.7965  0.4258
total_pop_lag1                      -0.0010  0.0170  -0.0602  0.9520
lag_Violent_Crime_per_capita_lag1    0.1644  2.4323   0.0676  0.9461
lag_Property_Crime_per_capita_lag1  -3.7823  1.8311  -2.0656  0.0389
lag_minority_pct_lag1               -1.0830  0.7743  -1.3987  0.1619
lag_per_capita_income_lag1        

In [58]:
res = model_twfe_cluster 

def _rename_term(name: str) -> str:
    name = re.sub(r'^C\(Year\)\[T\.(\d+)\]$', r'Year_\1', name)
    name = re.sub(r'^C\(Precinct\)\[T\.(.+)\]$', r'Precinct_\1', name)
    if name == "Intercept":
        name = "const"
    return name

renamed_idx = [_rename_term(s) for s in res.params.index]
params_ren = pd.Series(res.params.values, index=renamed_idx, name="coef")
se_ren     = pd.Series(res.bse.values,     index=renamed_idx, name="se")
t_ren      = pd.Series(res.tvalues.values, index=renamed_idx, name="z")
p_ren      = pd.Series(res.pvalues.values, index=renamed_idx, name="pval")

table = pd.concat([params_ren, se_ren, t_ren, p_ren], axis=1)
table["is_FE"] = table.index.str.startswith(("Year_", "Precinct_"))

preferred_order = [
    "const",
    # Core outcome spillover + own lag(s)
    "lag_Budget_per_capita",
    "Budget_per_capita_lag1",
    "lag_Budget_per_capita_lag1",
    # Disaggregated index crimes 
    "MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY_ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND_LARCENY_per_capita_lag1",
    "GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    # Aggregated crimes 
    "Violent_Crime_per_capita_lag1",
    "Property_Crime_per_capita_lag1",
    # Socioeconomics
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1",
    # Spatial lags of X
    "lag_MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1",
    "lag_RAPE_per_capita_lag1",
    "lag_FELONY_ASSAULT_per_capita_lag1",
    "lag_BURGLARY_per_capita_lag1",
    "lag_GRAND_LARCENY_per_capita_lag1",
    "lag_GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1",
    "lag_ROBBERY_per_capita_lag1",
    "lag_Violent_Crime_per_capita_lag1",
    "lag_Property_Crime_per_capita_lag1",
    "lag_minority_pct_lag1",
    "lag_per_capita_income_lag1",
    "lag_total_pop_lag1",
]

substantive = table.loc[~table["is_FE"]].drop(columns="is_FE")

present_pref = [v for v in preferred_order if v in substantive.index]
remaining    = [v for v in substantive.index if v not in present_pref]
if remaining:
    remaining_sorted = (
        substantive.loc[remaining]
                  .assign(_abs=lambda s: s["coef"].abs())
                  .sort_values("_abs", ascending=False)
                  .drop(columns="_abs")
                  .index.tolist()
    )
else:
    remaining_sorted = []

ordered_index = present_pref + remaining_sorted
clean_table = substantive.loc[ordered_index].round(4)

print("\n=== TWFE with Spillovers — Substantive Coefficients (FE hidden) ===")
print(clean_table.to_string())

dw = durbin_watson(res.resid)
print("\n=== Model Fit (robust/clustered) ===")
print(f"Observations: {int(res.nobs)}")
print(f"R-squared / Adj.: {res.rsquared:.3f} / {res.rsquared_adj:.3f}")
if res.fvalue is not None:
    print(f"F-statistic (model): {res.fvalue:.2f}, p-value: {res.f_pvalue:.2g}")
print(f"Log-Likelihood: {res.llf:.1f} | AIC: {res.aic:.1f} | BIC: {res.bic:.1f}")
print(f"Durbin-Watson: {dw:.3f}")
print(f"Covariance Type: {res.cov_type} (groups=Precinct)")

n_year_fe = (table.index.str.startswith("Year_")).sum()
n_prec_fe = (table.index.str.startswith("Precinct_")).sum()
print(f"\n[Note] Hidden fixed effects: {n_year_fe} Year FEs, {n_prec_fe} Precinct FEs.")

def cleaned_summary_no_fe(res_):
    txt = res_.summary().as_text()
    # rename in the text first
    txt = re.sub(r'C\(Year\)\[T\.(\d+)\]', r'Year_\1', txt)
    txt = re.sub(r'C\(Precinct\)\[T\.([^\]]+)\]', r'Precinct_\1', txt)
    # drop any lines that start with Year_ or Precinct_
    cleaned_lines = []
    for line in txt.splitlines():
        if re.match(r'^\s*(Year_|Precinct_)', line):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)





=== TWFE with Spillovers — Substantive Coefficients (FE hidden) ===
                                       coef      se        z    pval
const                               29.3907  8.6410   3.4013  0.0007
lag_Budget_per_capita              -35.8373  5.3245  -6.7307  0.0000
Budget_per_capita_lag1               0.5428  0.0497  10.9237  0.0000
lag_Budget_per_capita_lag1          19.7627  4.0296   4.9044  0.0000
Violent_Crime_per_capita_lag1        0.0046  0.0436   0.1062  0.9155
Property_Crime_per_capita_lag1      -0.0500  0.0364  -1.3711  0.1703
minority_pct_lag1                   -0.0388  0.0208  -1.8675  0.0618
per_capita_income_lag1              -0.0190  0.0239  -0.7965  0.4258
total_pop_lag1                      -0.0010  0.0170  -0.0602  0.9520
lag_Violent_Crime_per_capita_lag1    0.1644  2.4323   0.0676  0.9461
lag_Property_Crime_per_capita_lag1  -3.7823  1.8311  -2.0656  0.0389
lag_minority_pct_lag1               -1.0830  0.7743  -1.3987  0.1619
lag_per_capita_income_lag1        

In [59]:
def _rename_term(name: str) -> str:
    name = re.sub(r'^C\(Year\)\[T\.(\d+)\]$', r'Year_\1', name)          
    name = re.sub(r'^C\(Precinct\)\[T\.(.+)\]$', r'Precinct_\1', name)   
    if name == "Intercept":
        name = "const"
    return name

def print_classic_summary_renamed(res):
    """
    Prints statsmodels' classic summary, but with FE names cleaned:
    - C(Year)[T.x]     -> Year_x
    - C(Precinct)[T.y] -> Precinct_y
    - Intercept        -> const
    Keeps *all* FE rows (nothing hidden).
    """
    summ = res.summary()           
    coef_tbl = summ.tables[1]     

    for i in range(1, len(coef_tbl.data)):  
        coef_tbl.data[i][0] = _rename_term(coef_tbl.data[i][0])

    print(summ)

res = model_twfe_cluster
print_classic_summary_renamed(res)


                            OLS Regression Results                            
Dep. Variable:      Budget_per_capita   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                     126.8
Date:                Mon, 13 Oct 2025   Prob (F-statistic):           6.13e-51
Time:                        01:25:24   Log-Likelihood:                 1191.5
No. Observations:                1241   AIC:                            -2179.
Df Residuals:                    1139   BIC:                            -1656.
Df Model:                         101                                         
Covariance Type:              cluster                                         
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
Inte

strong multicollinearity or other numerical problems.


In [60]:
res = model_twfe_cluster  

def _rename_term(name: str) -> str:
    name = re.sub(r'^C\(Year\)\[T\.(\d+)\]$', r'Year_\1', name)
    name = re.sub(r'^C\(Precinct\)\[T\.([^\]]+)\]$', r'Precinct_\1', name)
    if name == "Intercept":
        name = "const"
    return name

renamed_idx = [_rename_term(s) for s in res.params.index]
params_ren = pd.Series(res.params.values, index=renamed_idx, name="coef")
se_ren     = pd.Series(res.bse.values,     index=renamed_idx, name="se")
t_ren      = pd.Series(res.tvalues.values, index=renamed_idx, name="z")
p_ren      = pd.Series(res.pvalues.values, index=renamed_idx, name="pval")

table = pd.concat([params_ren, se_ren, t_ren, p_ren], axis=1)
table["is_FE"] = table.index.str.startswith(("Year_", "Precinct_"))

preferred_order = [
    "const",
    "lag_Budget_per_capita",
    "Budget_per_capita_lag1",
    "lag_Budget_per_capita_lag1",
    "MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1",
    "RAPE_per_capita_lag1",
    "FELONY_ASSAULT_per_capita_lag1",
    "BURGLARY_per_capita_lag1",
    "GRAND_LARCENY_per_capita_lag1",
    "GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1",
    "ROBBERY_per_capita_lag1",
    "Violent_Crime_per_capita_lag1",
    "Property_Crime_per_capita_lag1",
    "minority_pct_lag1",
    "per_capita_income_lag1",
    "total_pop_lag1",
    "lag_MURDER_NON_NEGL_MANSLAUGHTER_per_capita_lag1",
    "lag_RAPE_per_capita_lag1",
    "lag_FELONY_ASSAULT_per_capita_lag1",
    "lag_BURGLARY_per_capita_lag1",
    "lag_GRAND_LARCENY_per_capita_lag1",
    "lag_GRAND_LARCENY_OF_MOTOR_VEHICLE_per_capita_lag1",
    "lag_ROBBERY_per_capita_lag1",
    "lag_Violent_Crime_per_capita_lag1",
    "lag_Property_Crime_per_capita_lag1",
    "lag_minority_pct_lag1",
    "lag_per_capita_income_lag1",
    "lag_total_pop_lag1",
]

substantive = table.loc[~table["is_FE"]].drop(columns="is_FE")
present_pref = [v for v in preferred_order if v in substantive.index]
remaining    = [v for v in substantive.index if v not in present_pref]
remaining_sorted = []
if remaining:
    remaining_sorted = (
        substantive.loc[remaining]
                  .assign(_abs=lambda s: s["coef"].abs())
                  .sort_values("_abs", ascending=False)
                  .drop(columns="_abs")
                  .index.tolist()
    )
ordered_index = present_pref + remaining_sorted
clean_table = substantive.loc[ordered_index].round(4)

print("\n=== TWFE with Spillovers — Substantive Coefficients (FE hidden) ===")
print(clean_table.to_string())

dw = durbin_watson(res.resid)
print("\n=== Model Fit (robust/clustered) ===")
print(f"Observations: {int(res.nobs)}")
print(f"R-squared / Adj.: {res.rsquared:.3f} / {res.rsquared_adj:.3f}")
if res.fvalue is not None:
    print(f"F-statistic (model): {res.fvalue:.2f}, p-value: {res.f_pvalue:.2g}")
print(f"Log-Likelihood: {res.llf:.1f} | AIC: {res.aic:.1f} | BIC: {res.bic:.1f}")
print(f"Durbin-Watson: {dw:.3f}")
print(f"Covariance Type: {res.cov_type} (groups=Precinct)")

n_year_fe = (table.index.str.startswith("Year_")).sum()
n_prec_fe = (table.index.str.startswith("Precinct_")).sum()
print(f"\n[Note] Hidden fixed effects: {n_year_fe} Year FEs, {n_prec_fe} Precinct FEs.")

def relabel_exog_inplace(res_):
    """
    Overwrite model xnames so summary() prints without C(...).
    Clears caches so Series pick up new names.
    """
    # Try to get current names
    if hasattr(res_.model, "data") and getattr(res_.model.data, "xnames", None):
        xnames = list(res_.model.data.xnames)
    elif hasattr(res_.model, "exog_names") and res_.model.exog_names:
        xnames = list(res_.model.exog_names)
    else:
        return res_

    new_names = [_rename_term(n) for n in xnames]

    # Assign back
    if hasattr(res_.model, "exog_names"):
        res_.model.exog_names = new_names
    if hasattr(res_.model, "data") and hasattr(res_.model.data, "xnames"):
        res_.model.data.xnames = new_names
    if hasattr(res_.model, "xnames"):
        res_.model.xnames = new_names

    # Clear cached Series/tables
    if hasattr(res_, "_cache") and isinstance(res_._cache, dict):
        res_._cache.clear()

    return res_

def print_summary_keep_fe_noC(res_):
    """
    Classic statsmodels summary WITH all FE rows, but labels cleaned:
    C(Year)[T.x] -> Year_x; C(Precinct)[T.y] -> Precinct_y; Intercept -> const
    """
    txt = res_.summary().as_text()
    txt = re.sub(r'C\(Year\)\[T\.([^\]]+)\]', r'Year_\1', txt)
    txt = re.sub(r'C\(Precinct\)\[T\.([^\]]+)\]', r'Precinct_\1', txt)
    txt = txt.replace("Intercept", "const")
    print(txt)

def print_summary_drop_fe_noC(res_):
    """
    Classic statsmodels summary with FE rows REMOVED and labels cleaned.
    """
    txt = res_.summary().as_text()
    txt = re.sub(r'C\(Year\)\[T\.([^\]]+)\]', r'Year_\1', txt)
    txt = re.sub(r'C\(Precinct\)\[T\.([^\]]+)\]', r'Precinct_\1', txt)
    txt = txt.replace("Intercept", "const")
    cleaned_lines = []
    for line in txt.splitlines():
        if re.match(r'^\s*(Year_|Precinct_)', line):
            continue
        cleaned_lines.append(line)
    print("\n".join(cleaned_lines))




=== TWFE with Spillovers — Substantive Coefficients (FE hidden) ===
                                       coef      se        z    pval
const                               29.3907  8.6410   3.4013  0.0007
lag_Budget_per_capita              -35.8373  5.3245  -6.7307  0.0000
Budget_per_capita_lag1               0.5428  0.0497  10.9237  0.0000
lag_Budget_per_capita_lag1          19.7627  4.0296   4.9044  0.0000
Violent_Crime_per_capita_lag1        0.0046  0.0436   0.1062  0.9155
Property_Crime_per_capita_lag1      -0.0500  0.0364  -1.3711  0.1703
minority_pct_lag1                   -0.0388  0.0208  -1.8675  0.0618
per_capita_income_lag1              -0.0190  0.0239  -0.7965  0.4258
total_pop_lag1                      -0.0010  0.0170  -0.0602  0.9520
lag_Violent_Crime_per_capita_lag1    0.1644  2.4323   0.0676  0.9461
lag_Property_Crime_per_capita_lag1  -3.7823  1.8311  -2.0656  0.0389
lag_minority_pct_lag1               -1.0830  0.7743  -1.3987  0.1619
lag_per_capita_income_lag1        